## Integer Damath DynaQ

In [47]:
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict, Any
import numpy as np
import copy
from collections import defaultdict, deque


In [48]:
Operator = Optional[str]  # '+', '-', 'x', '/' or None

@dataclass
class Piece:
    player: int  # 1 = Blue, -1 = Red
    value: int
    dama: bool = False

    def __repr__(self):
        # Convert B to blue emoji and R to red emoji for better visualization
        p = "🔵" if self.player == 1 else "🔴"
        val = f"{self.value:+d}" if self.value >= 0 else str(self.value)
        return f"{p}{'D' if self.dama else ''}{val}"
    
    
    def copy(self):
        return Piece(self.player, self.value, self.dama)

@dataclass
class Move:
    path: List[Tuple[int, int]]            # sequence of positions traversed
    captures: List[Tuple[int, int]]        # list of captured piece positions
    promotes: bool = False                 # whether the move results in promotion
    score_gain: int = 0                    # arithmetic reward from the move
    is_dama_capture: bool = False          # whether move made by dama
    is_multi_jump: bool = False            # whether multiple captures occurred

    def __repr__(self):
        cap_str = f" x{len(self.captures)}" if self.captures else ""
        promo = " (promo)" if self.promotes else ""
        return f"{self.path}{cap_str}{promo} +{self.score_gain}"



In [49]:
class DamathEnv:
    def __init__(self, rows=8, cols=8, operator_pattern=None):
        self.R = rows
        self.C = cols
        assert rows==8 and cols==8, "Currently implemented for 8x8 boards."
        # Operators on playable squares. Default pattern similar to provided image if None.
        if operator_pattern is None:
            operator_pattern = self.default_operator_board()
        self.op_board = operator_pattern
        # Piece board: dict (r,c)->Piece
        self.pieces: Dict[Tuple[int,int], Piece] = {}
        # Scores cumulative per player
        self.scores = {1: 0.0, -1: 0.0}
        self.to_move = 1  # 1 starts (blue on top)
        self.history_states = deque(maxlen=50)  # for repetition detection (store simple board hashes)
        
        # initialize sample starting board if user wants. We'll provide a helper to set initial config.
        self.init_default_integer_setup()

    def default_operator_board(self):
        # Create operator layout (8x8) using a repeating pattern similar to the uploaded assets.
        # Operators placed on playable squares (r+c)%2==1.
        ops = ['x','/','-','+']  # cycle
        board = [[None for _ in range(self.C)] for __ in range(self.R)]
        for r in range(self.R):
            for c in range(self.C):
                if (r + c) % 2 == 1:
                    # choose operator based on some pattern; rotate every cell
                    board[r][c] = ops[(r + 2*c) % len(ops)]
                else:
                    board[r][c] = None
        return board

    def init_default_integer_setup(self):
        # Initialize pieces according to the "integer damath" sample. We'll follow a symmetric-ish layout.
        # Blue (player=1) on top three rows playable squares, Red (player=-1) on bottom three rows.
        self.pieces = {}
        # sample integer values, you can customize to exact image mapping
        blue_values = [
            [-11, 8, -5, 2],
            [0, -3, 10, -7],
            [-9, 6, -1, 4],
        ]
        red_values = [
            [4, -1, 6, -9],
            [-7, 10, -3, 0],
            [2, -5, 8, -11]
        ]
        # place on playable squares; for top rows choose columns 0,2,4,6 for row 0,1.. pattern.
        # We'll place blues on rows 0..2 and reds on rows 5..7 so that they face each other.
        # map values left to right
        def playable_positions_on_row(r):
            # playable cols where (r+c)%2==1
            return [c for c in range(self.C) if (r+c)%2==1]
        # Blue top 3 rows
        for i, r in enumerate(range(0,3)):
            cols = playable_positions_on_row(r)
            vals = blue_values[i]
            for j, c in enumerate(cols[:len(vals)]):
                self.pieces[(r,c)] = Piece(player=1, value=vals[j], dama=False)
        # Red bottom 3 rows
        for i, r in enumerate(range(5,8)):
            cols = playable_positions_on_row(r)
            vals = red_values[i-0] if i < len(red_values) else []
            for j, c in enumerate(cols[:len(vals)]):
                self.pieces[(r,c)] = Piece(player=-1, value=vals[j], dama=False)
        # reset scores and to_move
        self.scores = {1:0.0, -1:0.0}
        self.to_move = 1
        self.history_states.clear()
        self.record_state()

    def copy(self):
        newenv = DamathEnv(self.R, self.C)
        newenv.op_board = copy.deepcopy(self.op_board)
        newenv.pieces = {k: v.copy() for k,v in self.pieces.items()}
        newenv.scores = dict(self.scores)
        newenv.to_move = self.to_move
        newenv.history_states = copy.deepcopy(self.history_states)
        return newenv

    def in_bounds(self, r,c):
        return 0 <= r < self.R and 0 <= c < self.C

    def is_playable(self, r,c):
        return self.in_bounds(r,c) and ((r+c)%2==1)

    def get_piece(self, r,c) -> Optional[Piece]:
        return self.pieces.get((r,c))

    def remove_piece(self, r,c):
        if (r,c) in self.pieces:
            del self.pieces[(r,c)]

    def move_piece(self, from_rc, to_rc):
        p = self.pieces.pop(from_rc)
        self.pieces[to_rc] = p
        return p

    def record_state(self):
        # Simple hash of pieces positions and values and to_move for repetition detection
        items = tuple(sorted([ (pos, piece.player, piece.value, piece.dama) for pos,piece in self.pieces.items() ]))
        key = (self.to_move, items)
        self.history_states.append(key)

    # ----------------------------- Operators & arithmetic -----------------------------
    def op_at(self, r,c):
        if not self.is_playable(r,c):
            return None
        return self.op_board[r][c]

    def apply_operator(self, op: str, a: int, b: int):
        if op == '+':
            return a + b
        if op == '-':
            return a - b
        if op == 'x' or op == 'X' or op == '*':
            return a * b
        if op == '/':
            # integer division semantics: handle division by zero and prefer integer division rounding toward zero
            if b == 0:
                # define a penalty or large negative? For now, return 0 to avoid crash.
                return 0
            return int(a / b)
        raise ValueError("Unknown op "+str(op))

    # ----------------------------- Movement and capture generation -----------------------------
    def generate_all_moves(self, player:int):
        """
        Returns a list of Move objects representing all legal moves for player.
        Enforces mandatory captures and priority rules described in prompt.
        """
        # 1) Find all capture sequences for every piece
        capture_moves = []
        for pos, piece in list(self.pieces.items()):
            if piece.player != player: continue
            caps = self._generate_captures_from(pos, piece)
            capture_moves.extend(caps)
        if len(capture_moves) > 0:
            # enforce capture priority: (1) max captures, (2) if tie, dama priority, (3) if regular has more captures than dama, regular wins
            max_cap = max(len(m.captures) for m in capture_moves)
            # filter moves with max captures
            maxcap_moves = [m for m in capture_moves if len(m.captures)==max_cap]
            # among tied moves, if any are from dama pieces and any from regular, prefer dama (unless regular has strictly more captures in other moves)
            # but since we already filtered to max_cap, only need to prefer dama among ties => prefer moves with piece.dama True if any
            if any(self.pieces[m.path[0]].dama for m in maxcap_moves): # note m.path[0] original square
                maxcap_moves = [m for m in maxcap_moves if self.pieces[m.path[0]].dama]
            # compute score_gain for each move using current op board and multipliers
            for m in maxcap_moves:
                m.score_gain = self._compute_move_score(m, player)
            return maxcap_moves
        # 2) If no captures, generate simple moves including dama moves
        simple_moves = []
        for pos, piece in list(self.pieces.items()):
            if piece.player != player: continue
            sms = self._generate_simple_from(pos, piece)
            simple_moves.extend(sms)
        # mark score_gain zero for these
        for m in simple_moves:
            m.score_gain = 0.0
        return simple_moves

    def _generate_simple_from(self, pos, piece:Piece):
        r,c = pos
        moves = []
        if piece.dama:
            # dama can move any distance along diagonals (like king in international draughts)
            for dr,dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
                step=1
                while True:
                    nr = r + dr*step; nc = c + dc*step
                    if not self.in_bounds(nr,nc) or not self.is_playable(nr,nc): break
                    if (nr,nc) in self.pieces: break
                    path = [(r,c),(nr,nc)]
                    promotes = self._check_promotion(nr, piece.player)
                    moves.append(Move(path=path, captures=[], promotes=promotes))
                    step += 1
        else:
            # regular piece: forward-only? In Damath, regular pieces can move diagonally forward one space.
            # We assume player=1 moves 'down' (increasing row), player=-1 moves 'up' (decreasing row).
            dr = 1 if piece.player==1 else -1
            for dc in (-1,1):
                nr = r + dr; nc = c + dc
                if not self.in_bounds(nr,nc) or not self.is_playable(nr,nc): continue
                if (nr,nc) in self.pieces: continue
                path=[(r,c),(nr,nc)]
                promotes = self._check_promotion(nr, piece.player)
                moves.append(Move(path=path, captures=[], promotes=promotes))
        return moves

    def _generate_captures_from(self, pos, piece:Piece):
        # returns all capture sequences starting from this piece (as Move objects)
        # For regular pieces: jump over adjacent opponent piece landing on square beyond if empty; can chain.
        # For dama pieces: long-range capture along diagonals: can jump over an opponent piece that has at least one empty landing square beyond it on the same diagonal. Dama can land on any empty square beyond the captured piece on that diagonal (but rules about priority when multiple captures available are handled globally).
        results = []
        r,c = pos

        if piece.dama:
            # long-range captures: for each diagonal, find opponent pieces and possible landing squares beyond.
            for dr,dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
                # step along diagonal to find first opponent piece(s)
                step=1
                while True:
                    mr = r + dr*step; mc = c + dc*step
                    if not self.in_bounds(mr,mc) or not self.is_playable(mr,mc): break
                    if (mr,mc) in self.pieces:
                        target = self.pieces[(mr,mc)]
                        if target.player == piece.player:
                            break  # blocked by own piece
                        # find landing squares beyond (must be empty)
                        land_step = 1
                        while True:
                            lr = mr + dr*land_step; lc = mc + dc*land_step
                            if not self.in_bounds(lr,lc) or not self.is_playable(lr,lc): break
                            if (lr,lc) in self.pieces: break
                            # found a possible landing square
                            # create a tentative move: capture that one piece and land on (lr,lc)
                            new_env = self.copy()
                            # perform capture on new_env to continue searching for multi-captures
                            captured_piece = new_env.pieces.pop((mr,mc))
                            moved_piece = new_env.pieces.pop((r,c))
                            new_env.pieces[(lr,lc)] = moved_piece
                            # Recurse to find further captures from (lr,lc)
                            # Note: we store the captured piece object snapshot as part of capture tuple
                            further = new_env._generate_captures_from((lr,lc), moved_piece)
                            if len(further)==0:
                                m = Move(path=[(r,c),(lr,lc)], captures=[(mr,mc,captured_piece)], promotes=False)
                                results.append(m)
                            else:
                                for fm in further:
                                    # prepend current capture to fm
                                    m = Move(path=[(r,c)] + fm.path, captures=[(mr,mc,captured_piece)] + fm.captures, promotes=False)
                                    results.append(m)
                            land_step += 1
                        break  # only consider the first opponent piece along diagonal for long-range capture
                    else:
                        step += 1

        else:
            # regular piece captures: check adjacent diagonals for opponent piece and landing square beyond
            for dr,dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
                ar = r + dr; ac = c + dc  # adjacent
                lr = r + 2*dr; lc = c + 2*dc  # landing beyond
                if not self.in_bounds(ar,ac) or not self.is_playable(ar,ac): continue
                if (ar,ac) not in self.pieces: continue
                if self.pieces[(ar,ac)].player == piece.player: continue
                if not self.in_bounds(lr,lc) or not self.is_playable(lr,lc): continue
                if (lr,lc) in self.pieces: continue
                # simulate capture and recurse
                new_env = self.copy()
                captured_piece = new_env.pieces.pop((ar,ac))
                moved_piece = new_env.pieces.pop((r,c))
                new_env.pieces[(lr,lc)] = moved_piece
                further = new_env._generate_captures_from((lr,lc), moved_piece)
                if len(further)==0:
                    m = Move(path=[(r,c),(lr,lc)], captures=[(ar,ac,captured_piece)], promotes=self._check_promotion(lr,piece.player))
                    results.append(m)
                else:
                    for fm in further:
                        m = Move(path=[(r,c)] + fm.path, captures=[(ar,ac,captured_piece)] + fm.captures, promotes=self._check_promotion(fm.path[-1][0], piece.player))
                        results.append(m)
        # remove duplicate sequences (same path) - dedupe by path and captured coordinates
        uniq = {}
        for m in results:
            key = (tuple(m.path), tuple((r,c,cp.value) for r,c,cp in m.captures))
            if key not in uniq or len(uniq[key].captures) < len(m.captures):
                uniq[key] = m
        return list(uniq.values())

    def _compute_move_score(self, move:Move, mover_player:int):
        # compute arithmetic score gain to mover for a capture move following rules:
        # - each captured piece adds op(own_value, captured_value) where op is operator on the landing square of that jump
        # - for dama captures, double score; if both dama, quadruple for that take.
        # - if a dama is taken by regular, the score is doubled as well (we handle multiplier on capture event)
        total = 0.0
        # need to reconstruct the piece value used in each jump: the moving piece's value at that time can be assumed unchanged
        # We'll use the mover's piece's original value
        # For multi-jumps, landing squares determine operators used
        mover_start = move.path[0]
        mover_piece = self.pieces.get(mover_start)
        mover_is_dama = mover_piece.dama if mover_piece else False
        # We must approximate the moving piece's value; in rules it's the mover's own piece value each time
        mover_value = mover_piece.value if mover_piece else 0
        # For each capture in sequence, determine operator at landing square (the square after the specific jump)
        # To find landing square for ith capture: it's path[1+i]
        for i, (cap_r,cap_c,cap_piece) in enumerate(move.captures):
            landing = move.path[1+i] if len(move.path) > 1+i else move.path[-1]
            op = self.op_at(*landing)
            # apply operator: op(self_value, captured_value)
            base = self.apply_operator(op, mover_value, cap_piece.value)
            # multipliers:
            mult = 1
            # If mover is dama, double for that take; if captured is dama and mover is dama -> quadruple (2*2)
            if mover_is_dama and cap_piece.dama:
                mult = 4
            elif mover_is_dama:
                mult = 2
            elif cap_piece.dama:
                # captured is dama and mover is regular: score doubled as well per rules
                mult = 2
            total += base * mult
        return total

    def _check_promotion(self, r, player):
        # If a piece reaches opposing end (row 7 for player=1, row 0 for player=-1), it is promoted
        if player==1 and r==self.R-1: return True
        if player==-1 and r==0: return True
        return False

    # ----------------------------- Apply Move -----------------------------
    def apply_move(self, move: "Move", verbose=False):
        """
        Apply a move and update board state + cumulative self.scores.
        Immediate reward is disabled — returns 0.0 every call.
        """
        player = self.to_move

        if len(move.captures) == 0:
            frm, to = move.path[0], move.path[-1]
            piece = self.pieces.pop(frm)
            self.pieces[to] = piece

            if self._check_promotion(to[0], piece.player) and not piece.dama:
                piece.dama = True

        else:
            frm = move.path[0]
            mover = self.pieces.pop(frm)
            current_pos = frm
            total_gain = 0.0

            for i, (cap_r, cap_c, cap_piece_snapshot) in enumerate(move.captures):
                landing = move.path[1 + i] if 1 + i < len(move.path) else move.path[-1]
                op = self.op_at(*landing)
                captured_piece = self.pieces.pop((cap_r, cap_c))
                base = self.apply_operator(op, mover.value, captured_piece.value)

                mult = 1
                if mover.dama and captured_piece.dama:
                    mult = 4
                elif mover.dama or captured_piece.dama:
                    mult = 2

                total_gain += base * mult
                current_pos = landing

            self.pieces[current_pos] = mover
            if self._check_promotion(current_pos[0], mover.player) and not mover.dama:
                mover.dama = True
            self.scores[mover.player] += total_gain

        self.to_move *= -1
        self.record_state()

        if verbose:
            print(f"[Player {player}] applied move, scores={self.scores}")
        return 0.0
        
            # ----------------------------- Terminal Reward -----------------------------
    def compute_final_reward_for(self, player: int = 1, alpha: float = 0.7, K: float = 100.0):
        """
        Compute final combined reward for `player`:
            final_reward = α * binary_outcome + (1 - α) * tanh((score_diff)/K)
        Returns (final_reward, winner, final_scores)
        """
        final_scores, winner = self.final_scores_and_winner()
        p_score = final_scores.get(player, 0.0)
        o_score = final_scores.get(-player, 0.0)

        # Binary component
        if winner == player:
            binary_reward = 1.0
        elif winner == -player:
            binary_reward = -1.0
        else:
            binary_reward = 0.0

        # Normalized score difference
        norm_diff = float(np.tanh((p_score - o_score) / K))
        final_reward = float(alpha * binary_reward + (1 - alpha) * norm_diff)
        final_reward = float(np.clip(final_reward, -1.0, 1.0))
        return final_reward, winner, final_scores


    # ----------------------------- Game end and scoring -----------------------------
    def legal_moves_exist(self, player:int):
        return len(self.generate_all_moves(player))>0

    def game_over(self):
        # game over if current player to move has no moves or only one player's chips remain or repetition or stalemate
        if not self.legal_moves_exist(self.to_move):
            return True
        # if only chips of one player remain
        players_present = set(p.player for p in self.pieces.values())
        if len(players_present) <= 1:
            return True
        # repetition detection (simple): if last 6 states repeated pattern
        # For now, consider repetition if history has same state repeated >=4 times overall
        hist = list(self.history_states)
        if len(hist) >= 8:
            counts = defaultdict(int)
            for h in hist:
                counts[h] += 1
                if counts[h] >= 4:
                    return True
        return False

    def final_scores_and_winner(self):
        # Add remaining pieces to their player's cumulative scores (dama doubled)
        final_scores = dict(self.scores)
        for (r,c), piece in self.pieces.items():
            val = piece.value * (2 if piece.dama else 1)
            final_scores[piece.player] += val
        # Determine winner
        if final_scores[1] > final_scores[-1]:
            winner = 1
        elif final_scores[1] < final_scores[-1]:
            winner = -1
        else:
            winner = 0
        return final_scores, winner

    # ----------------------------- Utilities -----------------------------
    def print_board(self):
        # Create board grid with operators and pieces
        grid = [[" ." for _ in range(self.C)] for __ in range(self.R)]
        for y in range(self.R):
            for x in range(self.C):
                if not self.is_playable(y, x):
                    grid[y][x] = "##"
                else:
                    op = self.op_at(y, x)
                    grid[y][x] = f" {op}"
        # Place pieces
        for (y, x), piece in self.pieces.items():
            sym = '🔵' if piece.player == 1 else '🔴'
            if piece.dama:
                sym += 'K'
            grid[y][x] = f"{sym}{piece.value:02d}" if piece.value >= 0 else f"{sym}{piece.value}"

        # Print column headers
        print("\n     " + " ".join([f"{x:>4}" for x in range(self.C)]))
        print("     " + "----" * self.C)

        # Print from top (highest y) to bottom (y=0)
        for y in reversed(range(self.R)):
            row_str = " ".join(f"{cell:>4}" for cell in grid[y])
            print(f"{y:>2} | {row_str} | {y:>2}")

        print("     " + "----" * self.C)
        print("     " + " ".join([f"{x:>4}" for x in range(self.C)]))


In [50]:
# Operator layout: (y, x) format
# y=0 is bottom row, y=7 is top row

operator_pattern_official = [
    ['x', '-', '/', 'x', '-', '+', '+', 'x'],
    ['-', '/', '-', 'x', '-', '+', 'x', '-'],
    ['-', '+', '+', '+', 'x', 'x', '/', '+'],
    ['x', '+', '+', '-', 'x', '/', '+', 'x'],
    ['x', '-', '/', 'x', '-', '-', '+', 'x'],
    ['-', '/', 'x', 'x', '-', '+', 'x', '-'],
    ['-', 'x', '+', '+', 'x', 'x', '/', '+'],
    ['+', '/', '-', '-', 'x', '/', '+', 'x']
]

env = DamathEnv(rows = 8, cols = 8, operator_pattern=operator_pattern_official)
env.print_board()
moves = env.generate_all_moves(env.to_move)
print("\nAvailable moves for player", env.to_move, "->", len(moves))
for move in moves:
    start = move.path[0]
    end = move.path[-1]
    piece = env.pieces.get(start, None)
    if piece:
        piece_type = "Dama" if piece.dama else "Regular"
        # Swap (y, x) to (x, y) for readability
        print(f"{piece_type} {piece.value:+d} at ({start[1]}, {start[0]}) -> ({end[1]}, {end[0]}) | ΔScore: {move.score_gain:+.1f}")
    else:
        print(f"Unknown piece at ({start[1]}, {start[0]}) -> ({end[1]}, {end[0]}) | ΔScore: {move.score_gain:+.1f}")




        0    1    2    3    4    5    6    7
     --------------------------------
 7 |  🔴02   ##  🔴-5   ##  🔴08   ## 🔴-11   ## |  7
 6 |   ##  🔴-7   ##  🔴10   ##  🔴-3   ##  🔴00 |  6
 5 |  🔴04   ##  🔴-1   ##  🔴06   ##  🔴-9   ## |  5
 4 |   ##    -   ##    x   ##    -   ##    x |  4
 3 |    x   ##    +   ##    x   ##    +   ## |  3
 2 |   ##  🔵-9   ##  🔵06   ##  🔵-1   ##  🔵04 |  2
 1 |  🔵00   ##  🔵-3   ##  🔵10   ##  🔵-7   ## |  1
 0 |   ## 🔵-11   ##  🔵08   ##  🔵-5   ##  🔵02 |  0
     --------------------------------
        0    1    2    3    4    5    6    7

Available moves for player 1 -> 7
Regular -9 at (1, 2) -> (0, 3) | ΔScore: +0.0
Regular -9 at (1, 2) -> (2, 3) | ΔScore: +0.0
Regular +6 at (3, 2) -> (2, 3) | ΔScore: +0.0
Regular +6 at (3, 2) -> (4, 3) | ΔScore: +0.0
Regular -1 at (5, 2) -> (4, 3) | ΔScore: +0.0
Regular -1 at (5, 2) -> (6, 3) | ΔScore: +0.0
Regular +4 at (7, 2) -> (6, 3) | ΔScore: +0.0


In [51]:
operator_pattern_official = [
    ['x', '+', '/', '-', '-', '/', '+', 'x'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['x', '+', '/', '-', '-', '/', '+', 'x'],
    ['x', '+', '/', '-', '-', '/', '+', 'x'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['x', '/', '/', '-', '-', '/', '+', 'x']
]

In [52]:
# Print empty board with no pieces, just operators
env_empty = DamathEnv(rows = 8, cols = 8, operator_pattern=operator_pattern_official)
env_empty.pieces = {}  # clear pieces
env_empty.print_board()


        0    1    2    3    4    5    6    7
     --------------------------------
 7 |    x   ##    /   ##    -   ##    +   ## |  7
 6 |   ##    /   ##    x   ##    +   ##    - |  6
 5 |    -   ##    +   ##    x   ##    /   ## |  5
 4 |   ##    +   ##    -   ##    /   ##    x |  4
 3 |    x   ##    /   ##    -   ##    +   ## |  3
 2 |   ##    /   ##    x   ##    +   ##    - |  2
 1 |    -   ##    +   ##    x   ##    /   ## |  1
 0 |   ##    +   ##    -   ##    /   ##    x |  0
     --------------------------------
        0    1    2    3    4    5    6    7


In [53]:
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

# create a unique run folder
run_name = f"runs/damath_selfplay_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(run_name)

In [54]:
from collections import deque
import random

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def add(self, examples):
        """Add list of examples [(state, pi, z, mover), ...]"""
        self.buffer.extend(examples)

    def sample(self, batch_size):
        """Randomly sample a batch of examples"""
        batch = random.sample(self.buffer, batch_size)
        return batch

    def __len__(self):
        return len(self.buffer)


In [55]:
import math
import random
import time
import copy
from collections import defaultdict, deque
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

# ----------------------- Playable squares mapping ------------------------
PLAYABLE_POS = [(r, c) for r in range(8) for c in range(8) if (r + c) % 2 == 1]
POS_TO_IDX = {pos: i for i, pos in enumerate(PLAYABLE_POS)}
IDX_TO_POS = {i: pos for pos, i in POS_TO_IDX.items()}
ACTION_SIZE = len(PLAYABLE_POS) * len(PLAYABLE_POS)

def move_to_index(move):
    start = move.path[0]
    end = move.path[-1]
    s_idx = POS_TO_IDX[start]
    e_idx = POS_TO_IDX[end]
    return s_idx * len(PLAYABLE_POS) + e_idx

def index_to_move_index_pair(idx):
    n = len(PLAYABLE_POS)
    s = idx // n
    e = idx % n
    return s, e

def encode_state(env):
    """Encode state as 9-channel tensor"""
    C, H, W = 9, 8, 8
    state = np.zeros((C, H, W), dtype=np.float32)
    
    for (y, x), piece in env.pieces.items():
        if piece.player == 1:
            state[1 if piece.dama else 0, y, x] = 1.0
            state[4, y, x] = piece.value / 12.0
        else:
            state[3 if piece.dama else 2, y, x] = 1.0
            state[5, y, x] = piece.value / 12.0
    
    op_map = {'+': 0.25, '-': 0.5, 'x': 0.75, 'X': 0.75, '*': 0.75, '/': 1.0, '÷': 1.0}
    for y in range(8):
        for x in range(8):
            if env.is_playable(y, x):
                state[6, y, x] = op_map.get(env.op_board[y][x], 0.0)
    
    state[7, :, :] = 1.0 if env.to_move == 1 else 0.0
    blue = env.scores.get(1, 0.0)
    red = env.scores.get(-1, 0.0)
    state[8, :, :] = (blue - red) / 100.0
    return state

# ----------------------- Q-Network ------------------------
class QNetwork(nn.Module):
    def __init__(self, in_ch=9, board_h=8, board_w=8, action_size=ACTION_SIZE):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(128)
        self.bn3 = nn.BatchNorm2d(128)
        
        # Q-value head
        self.q_conv = nn.Conv2d(128, 64, kernel_size=1)
        self.q_fc1 = nn.Linear(64 * board_h * board_w, 512)
        self.q_fc2 = nn.Linear(512, action_size)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        
        q = F.relu(self.q_conv(x))
        q = q.view(q.size(0), -1)
        q = F.relu(self.q_fc1(q))
        q = self.q_fc2(q)
        return q

# ----------------------- Model Network ------------------------
class ModelNetwork(nn.Module):
    def __init__(self, in_ch=9, board_h=8, board_w=8, action_size=ACTION_SIZE):
        super().__init__()
        
        # State encoder
        self.state_conv1 = nn.Conv2d(in_ch, 64, kernel_size=3, padding=1)
        self.state_conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        
        # Action embedding
        self.action_embed = nn.Embedding(action_size, 128)
        
        # Combined prediction
        self.pred_conv1 = nn.Conv2d(256, 128, kernel_size=3, padding=1)
        self.pred_conv2 = nn.Conv2d(128, 64, kernel_size=3, padding=1)
        self.pred_conv3 = nn.Conv2d(64, in_ch, kernel_size=3, padding=1)
        
        # Reward prediction
        self.reward_fc1 = nn.Linear(128 * board_h * board_w, 256)
        self.reward_fc2 = nn.Linear(256, 1)

    def forward(self, state, action_idx):
        B = state.size(0)
        
        # Encode state
        s = F.relu(self.state_conv1(state))
        s = F.relu(self.state_conv2(s))
        
        # Encode action
        a = self.action_embed(action_idx)
        a = a.view(B, 128, 1, 1).expand(-1, -1, 8, 8)
        
        # Concatenate
        combined = torch.cat([s, a], dim=1)
        
        # Predict next state
        next_state = F.relu(self.pred_conv1(combined))
        next_state = F.relu(self.pred_conv2(next_state))
        next_state = self.pred_conv3(next_state)
        
        # Predict reward
        flat = s.view(B, -1)
        reward = F.relu(self.reward_fc1(flat))
        reward = self.reward_fc2(reward).squeeze(-1)
        
        return next_state, reward

# ----------------------- Dyna-Q Agent ------------------------
class DynaQAgent:
    def __init__(self, q_net, model_net, model_optimizer, epsilon=0.1, alpha=0.001, 
                 gamma=0.95, planning_steps=10):
        """
        Args:
            q_net: Q-Network for value estimation
            model_net: Model Network for environment dynamics
            model_optimizer: Optimizer for the model network (IMPORTANT!)
            epsilon: Exploration rate
            alpha: Q-network learning rate
            gamma: Discount factor
            planning_steps: Number of planning steps per real step
        """
        self.q_net = q_net
        self.model_net = model_net
        self.model_optimizer = model_optimizer  # ✅ ADD THIS
        self.epsilon = epsilon
        self.alpha = alpha
        self.gamma = gamma
        self.planning_steps = planning_steps
        
        self.q_optimizer = optim.Adam(q_net.parameters(), lr=alpha)
        
        # Experience buffer
        self.experience_buffer = deque(maxlen=10000)

    def get_legal_mask(self, env):
        """Create mask for legal actions"""
        legal_moves = env.generate_all_moves(env.to_move)
        legal_indices = [move_to_index(m) for m in legal_moves]
        mask = np.zeros(ACTION_SIZE, dtype=np.float32)
        mask[legal_indices] = 1.0
        return mask, legal_moves, legal_indices

    def select_action(self, env, training=True):
        """Epsilon-greedy action selection"""
        mask, legal_moves, legal_indices = self.get_legal_mask(env)
        
        if not legal_moves:
            return None, None
        
        # Epsilon-greedy
        if training and random.random() < self.epsilon:
            chosen_idx = random.choice(legal_indices)
        else:
            state = encode_state(env)
            state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(next(self.q_net.parameters()).device)
            
            with torch.no_grad():
                q_values = self.q_net(state_tensor).squeeze(0).cpu().numpy()
            
            q_values = q_values * mask - (1 - mask) * 1e9
            chosen_idx = np.argmax(q_values)
        
        # Convert to move
        s_idx, e_idx = index_to_move_index_pair(chosen_idx)
        start = IDX_TO_POS[s_idx]
        end = IDX_TO_POS[e_idx]
        
        chosen_move = next((m for m in legal_moves if m.path[0] == start and m.path[-1] == end), 
                          random.choice(legal_moves))
        
        return chosen_move, chosen_idx

    def train_model_step(self, state, action_idx, reward, next_state):
        """Train model on single transition - called immediately after each step"""
        device = next(self.model_net.parameters()).device
        
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        action_tensor = torch.tensor([action_idx], dtype=torch.long).to(device)
        next_state_tensor = torch.tensor(next_state, dtype=torch.float32).unsqueeze(0).to(device)
        reward_tensor = torch.tensor([reward], dtype=torch.float32).to(device)
        
        # Forward pass
        pred_next_state, pred_reward = self.model_net(state_tensor, action_tensor)
        
        # Calculate losses
        state_loss = F.mse_loss(pred_next_state, next_state_tensor)
        reward_loss = F.mse_loss(pred_reward, reward_tensor)
        loss = state_loss + reward_loss
        
        # Backward pass
        self.model_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model_net.parameters(), 1.0)
        self.model_optimizer.step()
        
        return loss.item()

    def update_q(self, state, action_idx, reward, next_state, done, next_legal_mask):
        """Q-learning update"""
        device = next(self.q_net.parameters()).device
        
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        next_state_tensor = torch.tensor(next_state, dtype=torch.float32).unsqueeze(0).to(device)
        action_tensor = torch.tensor([action_idx], dtype=torch.long).to(device)
        reward_tensor = torch.tensor([reward], dtype=torch.float32).to(device)
        
        # Current Q
        current_q = self.q_net(state_tensor).gather(1, action_tensor.unsqueeze(1)).squeeze()
        
        # Target Q
        with torch.no_grad():
            if done:
                target_q = reward_tensor
            else:
                next_q_values = self.q_net(next_state_tensor).squeeze(0).cpu().numpy()
                next_q_values = next_q_values * next_legal_mask - (1 - next_legal_mask) * 1e9
                max_next_q = torch.tensor([np.max(next_q_values)], dtype=torch.float32).to(device)
                target_q = reward_tensor + self.gamma * max_next_q
        
        # Update
        loss = F.mse_loss(current_q, target_q)
        self.q_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 1.0)
        self.q_optimizer.step()
        
        return loss.item()

    def planning(self):
        """Dyna-Q planning step - use learned model to generate synthetic experience"""
        if len(self.experience_buffer) < 32:
            return 0.0
        
        total_loss = 0.0
        device = next(self.model_net.parameters()).device
        
        for _ in range(self.planning_steps):
            # Sample random experience
            state, action_idx, _, _, _ = random.choice(self.experience_buffer)
            
            state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
            action_tensor = torch.tensor([action_idx], dtype=torch.long).to(device)
            
            # Use model to predict next state and reward
            with torch.no_grad():
                pred_next_state, pred_reward = self.model_net(state_tensor, action_tensor)
                pred_next_state = pred_next_state.squeeze(0).cpu().numpy()
                pred_reward = pred_reward.item()
            
            # Update Q-network with synthetic experience
            dummy_mask = np.ones(ACTION_SIZE, dtype=np.float32)
            loss = self.update_q(state, action_idx, pred_reward, pred_next_state, False, dummy_mask)
            total_loss += loss
        
        return total_loss / self.planning_steps

    def store_experience(self, state, action_idx, reward, next_state, done):
        """Store experience and train model immediately"""
        self.experience_buffer.append((state, action_idx, reward, next_state, done))
        # ✅ Train model on this transition
        self.train_model_step(state, action_idx, reward, next_state)

# ----------------------- Self-Play Game ------------------------
def self_play_game(env_factory, agent, max_moves=300):
    """Play one game"""
    env = env_factory()
    move_count = 0
    experiences = []
    
    while not env.game_over() and move_count < max_moves:
        state = encode_state(env)
        legal_mask, _, _ = agent.get_legal_mask(env)
        
        move, action_idx = agent.select_action(env, training=True)
        if move is None:
            break
        
        old_scores = dict(env.scores)
        env.apply_move(move)
        new_scores = dict(env.scores)
        
        # Immediate reward
        reward = (new_scores[1] - old_scores[1]) - (new_scores[-1] - old_scores[-1])
        reward = reward / 100.0
        
        next_state = encode_state(env)
        next_legal_mask, _, _ = agent.get_legal_mask(env)
        done = env.game_over()
        
        # Store experience (this also trains the model)
        agent.store_experience(state, action_idx, reward, next_state, done)
        experiences.append((state, action_idx, reward, next_state, done))
        
        # Update Q-network
        agent.update_q(state, action_idx, reward, next_state, done, next_legal_mask)
        
        # Planning: use model to generate synthetic experiences
        agent.planning()
        
        move_count += 1
    
    # Final rewards
    final_reward_p1, winner, final_scores = env.compute_final_reward_for(player=1)
    final_reward_p2, _, _ = env.compute_final_reward_for(player=-1)
    
    return {
        "experiences": experiences,
        "final_scores": final_scores,
        "winner": winner,
        "move_count": move_count,
        "reward_p1": final_reward_p1,
        "reward_p2": final_reward_p2,
    }

In [56]:
def train_model(model_net, optimizer, experiences, batch_size=64, epochs=4):
    """Train the world model with detailed epoch logging"""
    if len(experiences) == 0:
        return
    
    
    model_net.train()
    device = next(model_net.parameters()).device  # Get the device the model is on
    dataset_size = len(experiences)
    
    for epoch in range(epochs):
        random.shuffle(experiences)
        total_loss = 0.0
        num_batches = 0
        
        for i in range(0, len(experiences), batch_size):
            batch = experiences[i:i + batch_size]
            if len(batch) == 0:
                continue
            
            # Prepare batch data - handle both tuple and dict formats
            states = []
            actions = []
            next_states = []
            rewards = []
            
            for exp in batch:
                if isinstance(exp, dict):
                    # Dictionary format
                    state = exp['state']
                    action = exp['action_idx']
                    reward = exp['reward']
                    next_state = exp['next_state']
                else:
                    # Tuple format: (state, action_idx, reward, next_state, done)
                    state = exp[0]
                    action = exp[1]
                    reward = exp[2]
                    next_state = exp[3]
                
                # Convert to tensor if needed
                if not isinstance(state, torch.Tensor):
                    state = torch.FloatTensor(state)
                if not isinstance(next_state, torch.Tensor):
                    next_state = torch.FloatTensor(next_state)
                
                states.append(state)
                actions.append(action)
                rewards.append(reward)
                next_states.append(next_state)
            
            # Stack and move to device
            states = torch.stack(states).to(device)
            actions = torch.tensor(actions, dtype=torch.long).to(device)
            next_states = torch.stack(next_states).to(device)
            rewards = torch.tensor(rewards, dtype=torch.float32).to(device)
            
            # Forward pass
            pred_next_states, pred_rewards = model_net(states, actions)
            
            # Calculate losses
            state_loss = F.mse_loss(pred_next_states, next_states)
            reward_loss = F.mse_loss(pred_rewards.squeeze(), rewards)
            loss = state_loss + reward_loss
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
        
      
        state_loss = F.mse_loss(pred_next_states, next_states)    # ✅ State prediction
        reward_loss = F.mse_loss(pred_rewards.squeeze(), rewards) # ✅ Reward prediction
        
        print(f"    Epoch {epoch+1}/{epochs} | State Loss: {state_loss:.3f} | Reward Loss: {reward_loss:.3f}")


In [57]:
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
import torch
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
import time

def plot_training_metrics(history):
    """
    Generate comprehensive training visualizations matching the reference layout
    
    Args:
        history: dict containing training metrics over episodes/iterations
    """
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.3)
    
    fig.suptitle('Q-Learning Metrics - Damath', fontsize=16, fontweight='bold', y=0.98)
    
    # Determine if we have episode-level or iteration-level data
    if 'episodes' in history:
        episodes = history['episodes']
    else:
        # Use iterations for cumulative data (wins, etc.)
        episodes = list(range(1, len(history['p1_wins']) + 1))
    
    # Create separate episode list for per-game data (scores)
    if 'p1_scores' in history and len(history['p1_scores']) > 0:
        episodes_per_game = list(range(1, len(history['p1_scores']) + 1))
    else:
        episodes_per_game = episodes
    
    # Calculate summary statistics
    total_p1_wins = history['p1_wins'][-1] if isinstance(history['p1_wins'][-1], (int, float)) else sum(history['p1_wins'])
    total_p2_wins = history['p2_wins'][-1] if isinstance(history['p2_wins'][-1], (int, float)) else sum(history['p2_wins'])
    total_draws = history.get('draws', [0])[-1] if 'draws' in history else 0
    total_games = total_p1_wins + total_p2_wins + total_draws
    
    p1_winrate = (total_p1_wins / total_games * 100) if total_games > 0 else 0
    p2_winrate = (total_p2_wins / total_games * 100) if total_games > 0 else 0
    draw_rate = (total_draws / total_games * 100) if total_games > 0 else 0
    
    # 1. Cumulative Win Rate by P1 and P2 (Top Left)
    ax = fig.add_subplot(gs[0, 0])
    if 'p1_win_rate' in history and 'p2_win_rate' in history:
        p1_winrate_arr = history['p1_win_rate']
        p2_winrate_arr = history['p2_win_rate']
    else:
        total_games_arr = np.array(history['p1_wins']) + np.array(history['p2_wins']) + np.array(history.get('draws', [0]*len(history['p1_wins'])))
        total_games_arr = np.maximum(total_games_arr, 1)  # Avoid division by zero
        p1_winrate_arr = np.array(history['p1_wins']) / total_games_arr * 100
        p2_winrate_arr = np.array(history['p2_wins']) / total_games_arr * 100
    
    ax.plot(episodes, p1_winrate_arr, linewidth=2, color='#4A90E2', label='P1 (Blue)', alpha=0.8)
    ax.plot(episodes, p2_winrate_arr, linewidth=2, color='#E74C3C', label='P2 (Red)', alpha=0.8)
    ax.set_xlabel('Episode', fontsize=10)
    ax.set_ylabel('Win Rate (%)', fontsize=10)
    ax.set_title('Cumulative Win Rate by P1 and P2', fontsize=11, fontweight='bold')
    ax.legend(loc='best', framealpha=0.9, fontsize=9)
    ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
    ax.set_ylim([0, 100])
    
    # 2. Score Differential (Absolute Value) (Top Right)
    ax = fig.add_subplot(gs[0, 1])
    if 'score_diffs' in history:
        # Raw data with high variance (light scatter)
        abs_diffs = [abs(d) for d in history['score_diffs']]
        ax.scatter(episodes, abs_diffs, alpha=0.3, s=5, color='#95A5A6')
        
        # Moving average for trend
        window_size = min(100, len(episodes) // 10)
        if window_size > 0:
            ma_diffs = np.convolve(abs_diffs, np.ones(window_size)/window_size, mode='valid')
            ma_episodes = episodes[window_size-1:]
            ax.plot(ma_episodes, ma_diffs, linewidth=2.5, color='#9B59B6', 
                   label=f'{window_size}-Episode MA', alpha=0.9)
    
    ax.set_xlabel('Episode', fontsize=10)
    ax.set_ylabel('Score Diff', fontsize=10)
    ax.set_title('Score Differential (Absolute Value)', fontsize=11, fontweight='bold')
    ax.legend(loc='best', framealpha=0.9, fontsize=9)
    ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
    
    # 3. Episode Length (Middle Left)
    ax = fig.add_subplot(gs[1, 0])
    if 'episode_lengths' in history or 'avg_lengths' in history:
        lengths = history.get('episode_lengths', history.get('avg_lengths', []))
        
        # Moving average
        window_size = min(100, len(episodes) // 10)
        if window_size > 0:
            ma_lengths = np.convolve(lengths, np.ones(window_size)/window_size, mode='valid')
            ma_episodes = episodes[window_size-1:]
            ax.plot(ma_episodes, ma_lengths, linewidth=2, color='#E67E22', 
                   label=f'Length ({window_size}-Episode MA)', alpha=0.9)
        else:
            ax.plot(episodes, lengths, linewidth=2, color='#E67E22', alpha=0.9)
    
    ax.set_xlabel('Episode', fontsize=10)
    ax.set_ylabel('Internal Per Episode', fontsize=10)
    ax.set_title('Episode Length (AvgLength)', fontsize=11, fontweight='bold')
    ax.legend(loc='best', framealpha=0.9, fontsize=9)
    ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
    
    # 4. Final Scores (Middle Right) - FIXED
    ax = fig.add_subplot(gs[1, 1])
    if 'p1_scores' in history and 'p2_scores' in history:
        p1_scores = history['p1_scores']
        p2_scores = history['p2_scores']
        
        # Check if we have valid data
        if len(p1_scores) > 0 and len(p2_scores) > 0:
            # Raw scatter data - use episodes_per_game instead of episodes
            ax.scatter(episodes_per_game, p1_scores, alpha=0.3, s=5, color='#E8B4D9', label='P1 (Blue)')
            ax.scatter(episodes_per_game, p2_scores, alpha=0.3, s=5, color='#D4A5A5', label='P2 (Red)')
            
            # Moving averages
            window_size = min(20, max(1, len(episodes_per_game) // 10))  # Smaller window for better visualization
            if window_size > 0 and len(episodes_per_game) >= window_size:
                ma_p1 = np.convolve(p1_scores, np.ones(window_size)/window_size, mode='valid')
                ma_p2 = np.convolve(p2_scores, np.ones(window_size)/window_size, mode='valid')
                ma_episodes_game = episodes_per_game[window_size-1:]
                ax.plot(ma_episodes_game, ma_p1, linewidth=2.5, color='#4A90E2', 
                       label=f'P1 {window_size}-MA', alpha=0.9)
                ax.plot(ma_episodes_game, ma_p2, linewidth=2.5, color='#E74C3C', 
                       label=f'P2 {window_size}-MA', alpha=0.9)
        else:
            ax.text(0.5, 0.5, 'No score data available', 
                   ha='center', va='center', transform=ax.transAxes)
    else:
        ax.text(0.5, 0.5, 'Score tracking not enabled\n(p1_scores/p2_scores missing)', 
               ha='center', va='center', transform=ax.transAxes)
    
    ax.set_xlabel('Episode', fontsize=10)
    ax.set_ylabel('Score', fontsize=10)
    ax.set_title('Final Scores', fontsize=11, fontweight='bold')
    ax.legend(loc='best', framealpha=0.9, fontsize=9)
    ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
    
    # 5. Mean Absolute Reward (Bottom Left)
    ax = fig.add_subplot(gs[2, 0])
    if 'p1_rewards' in history and 'p2_rewards' in history:
        p1_rewards = history['p1_rewards']
        p2_rewards = history['p2_rewards']
        
        # Calculate absolute rewards per episode
        if isinstance(p1_rewards[0], list):
            p1_abs_rewards = [np.mean(np.abs(r)) if len(r) > 0 else 0 for r in p1_rewards]
            p2_abs_rewards = [np.mean(np.abs(r)) if len(r) > 0 else 0 for r in p2_rewards]
        else:
            p1_abs_rewards = [abs(r) for r in p1_rewards]
            p2_abs_rewards = [abs(r) for r in p2_rewards]
        
        # Raw data with scatter
        ax.scatter(episodes, p1_abs_rewards, alpha=0.3, s=5, color='#E8B4D9')
        ax.scatter(episodes, p2_abs_rewards, alpha=0.3, s=5, color='#D4A5A5')
        
        # Moving averages
        window_size = min(20, max(1, len(episodes) // 10))
        if window_size > 0 and len(episodes) >= window_size:
            ma_p1 = np.convolve(p1_abs_rewards, np.ones(window_size)/window_size, mode='valid')
            ma_p2 = np.convolve(p2_abs_rewards, np.ones(window_size)/window_size, mode='valid')
            ma_episodes = episodes[window_size-1:]
            ax.plot(ma_episodes, ma_p1, linewidth=2.5, color='#4A90E2', 
                   label=f'P1 {window_size}-MA', alpha=0.9)
            ax.plot(ma_episodes, ma_p2, linewidth=2.5, color='#E74C3C', 
                   label=f'P2 {window_size}-MA', alpha=0.9)
    
    ax.set_xlabel('Episode', fontsize=10)
    ax.set_ylabel('Reward', fontsize=10)
    ax.set_title('Mean Absolute Reward', fontsize=11, fontweight='bold')
    ax.legend(loc='best', framealpha=0.9, fontsize=9)
    ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
    
    # 6. Cumulative Wins per Player (Bottom Right)
    ax = fig.add_subplot(gs[2, 1])
    ax.plot(episodes, history['p1_wins'], linewidth=2.5, color='#4A90E2', 
            label='P1 (Blue) Wins', alpha=0.9)
    ax.plot(episodes, history['p2_wins'], linewidth=2.5, color='#E74C3C', 
            label='P2 (Red) Wins', alpha=0.9)
    
    ax.set_xlabel('Episode', fontsize=10)
    ax.set_ylabel('Cumulative Wins', fontsize=10)
    ax.set_title('Cumulative Wins per Player', fontsize=11, fontweight='bold')
    ax.legend(loc='best', framealpha=0.9, fontsize=9)
    ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
    
    plt.savefig('damath_training_metrics.png', dpi=300, bbox_inches='tight')
    print("📊 Training metrics saved to 'damath_training_metrics.png'")
    
    # Print comprehensive summary to console
    print("\n" + "="*100)
    print("TRAINING METRICS SUMMARY".center(100))
    print("="*100)
    print(f"\n{'TRAINING CONFIGURATION':^100}")
    print("-" * 100)
    print(f"Total Episodes:          {len(episodes)}")
    
    print(f"\n{'CUMULATIVE RESULTS':^100}")
    print("-" * 100)
    print(f"Total Games Played:      {total_games}")
    print(f"Player 1 Wins (Blue):    {total_p1_wins:>4} ({p1_winrate:>5.1f}%)")
    print(f"Player 2 Wins (Red):     {total_p2_wins:>4} ({p2_winrate:>5.1f}%)")
    print(f"Draws:                   {total_draws:>4} ({draw_rate:>5.1f}%)")
    
    # Episode metrics
    if 'episode_lengths' in history or 'avg_lengths' in history:
        lengths = history.get('episode_lengths', history.get('avg_lengths', []))
        print(f"\n{'EPISODE METRICS':^100}")
        print("-" * 100)
        print(f"Average Episode Length:  {np.mean(lengths):.2f} moves")
        print(f"Min Episode Length:      {min(lengths):.2f} moves")
        print(f"Max Episode Length:      {max(lengths):.2f} moves")
        print(f"Std Deviation:           {np.std(lengths):.2f} moves")
    
    # Reward metrics
    if 'p1_rewards' in history and history['p1_rewards']:
        p1_rewards_flat = [r for sublist in history['p1_rewards'] for r in (sublist if isinstance(sublist, list) else [sublist])]
        p2_rewards_flat = [r for sublist in history['p2_rewards'] for r in (sublist if isinstance(sublist, list) else [sublist])]
        
        print(f"\n{'REWARD METRICS':^100}")
        print("-" * 100)
        print(f"P1 Mean Reward:          {np.mean(p1_rewards_flat):+.4f}")
        print(f"P2 Mean Reward:          {np.mean(p2_rewards_flat):+.4f}")
    
    # Score differential
    if 'score_diffs' in history:
        print(f"\n{'SCORE DIFFERENTIAL (P1 - P2)':^100}")
        print("-" * 100)
        print(f"Average:                 {np.mean(history['score_diffs']):>+7.2f}")
        print(f"Std Deviation:           {np.std(history['score_diffs']):>7.2f}")
    
    # Score metrics if available
    if 'p1_scores' in history and 'p2_scores' in history:
        if len(history['p1_scores']) > 0:
            print(f"\n{'SCORE METRICS':^100}")
            print("-" * 100)
            print(f"P1 Average Score:        {np.mean(history['p1_scores']):.2f}")
            print(f"P2 Average Score:        {np.mean(history['p2_scores']):.2f}")
    
    print("\n" + "="*100 + "\n")
    
    plt.show()

In [58]:
def train_dynaq_with_viz(env_factory, num_episodes=200, games_per_iter=5, max_moves=200,
                         epsilon=0.2, alpha=0.001, gamma=0.9, planning_steps=10,
                         model_lr=0.001, model_batch_size=64, model_epochs=3,
                         buffer_size=10000, checkpoint_every=10):
    """Train Dyna-Q agent"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🎮 Using device: {device}")
    
    num_iterations = num_episodes // games_per_iter
    remaining_games = num_episodes % games_per_iter
    
    print(f"📊 Training Configuration:")
    print(f"   Total episodes: {num_episodes}")
    print(f"   Games per iteration: {games_per_iter}")
    print(f"   Number of iterations: {num_iterations}")
    
    # ✅ FIXED: Create networks and optimizer properly
    q_net = QNetwork().to(device)
    model_net = ModelNetwork().to(device)
    model_optimizer = optim.Adam(model_net.parameters(), lr=model_lr)
    
    # ✅ FIXED: Pass model_optimizer to agent
    agent = DynaQAgent(
        q_net=q_net,
        model_net=model_net,
        model_optimizer=model_optimizer,
        epsilon=epsilon,
        alpha=alpha,
        gamma=gamma,
        planning_steps=planning_steps
    )
    
    # TensorBoard setup
    run_name = f"runs/damath_dynaq_{datetime.now().strftime('%Y%m%d_%H%M%S')}_ep{num_episodes}"
    writer = SummaryWriter(run_name)
    
    # ✅ FIXED: Added p1_scores and p2_scores to history
    history = {
        'p1_wins': [],
        'p2_wins': [],
        'draws': [],
        'score_diffs': [],
        'avg_lengths': [],
        'p1_rewards': [],
        'p2_rewards': [],
        'p1_scores': [],  # ← ADDED
        'p2_scores': [],  # ← ADDED
    }
    
    global_step = 0
    cumulative_wins = {1: 0, -1: 0, 0: 0}
    
    # Training loop
    for iteration in range(1, num_iterations + 1):
        all_experiences = []
        game_lengths = []
        score_diffs = []
        win_counts = {1: 0, -1: 0, 0: 0}
        iter_p1_rewards = []
        iter_p2_rewards = []
        iter_p1_scores = []  # ← ADDED
        iter_p2_scores = []  # ← ADDED
        
        # Epsilon decay
        progress = global_step / num_episodes
        initial_epsilon = epsilon
        agent.epsilon = max(0.05, initial_epsilon * (1 - progress))
        
        games_this_iter = games_per_iter
        if iteration == num_iterations and remaining_games > 0:
            games_this_iter = remaining_games
        
        # Self-play games
        for g in range(games_this_iter):
            episode = self_play_game(env_factory, agent, max_moves)
            all_experiences.extend(episode["experiences"])
            
            winner = episode["winner"]
            win_counts[winner] += 1
            cumulative_wins[winner] += 1
            
            # ✅ FIXED: Collect final scores for each game
            final_p1 = episode["final_scores"].get(1, 0.0)
            final_p2 = episode["final_scores"].get(-1, 0.0)
            score_diff = final_p1 - final_p2
            
            iter_p1_scores.append(final_p1)  # ← ADDED
            iter_p2_scores.append(final_p2)  # ← ADDED
            
            game_lengths.append(episode["move_count"])
            score_diffs.append(score_diff)
            
            # Extract rewards
            current_player = 1
            for exp in episode["experiences"]:
                reward = float(exp[2])
                if current_player == 1:
                    iter_p1_rewards.append(reward)
                else:
                    iter_p2_rewards.append(-reward)
                current_player *= -1
            
            # Logging
            writer.add_scalar("Game/Length", episode["move_count"], global_step)
            writer.add_scalar("Game/Winner", winner, global_step)
            writer.add_scalar("Training/Epsilon", agent.epsilon, global_step)
            writer.add_scalar("Game/P1_Score", final_p1, global_step)  # ← ADDED
            writer.add_scalar("Game/P2_Score", final_p2, global_step)  # ← ADDED
            
            global_step += 1
            print(f"Episode {global_step}/{num_episodes} | Winner={winner} | "
                  f"Moves={episode['move_count']} | Score: P1={final_p1:.1f} P2={final_p2:.1f} | ε={agent.epsilon:.3f}")
        
        # ✅ FIXED: Store scores in history (extend for per-game data)
        history['p1_wins'].append(cumulative_wins[1])
        history['p2_wins'].append(cumulative_wins[-1])
        history['draws'].append(cumulative_wins[0])
        history['score_diffs'].append(float(np.mean(score_diffs)))
        history['avg_lengths'].append(float(np.mean(game_lengths)))
        history['p1_rewards'].append(iter_p1_rewards if iter_p1_rewards else [0.0])
        history['p2_rewards'].append(iter_p2_rewards if iter_p2_rewards else [0.0])
        
        # Extend score history with all games from this iteration
        history['p1_scores'].extend(iter_p1_scores)  # ← FIXED: use extend instead of append
        history['p2_scores'].extend(iter_p2_scores)  # ← FIXED: use extend instead of append
        
        # Optional: Additional batch model training
        if len(all_experiences) > model_batch_size:
            print(f"Additional model training on {len(all_experiences)} experiences...")
            train_model(model_net, model_optimizer, all_experiences, 
                       batch_size=model_batch_size, epochs=model_epochs)
        
        print(f"✅ Iter {iteration}/{num_iterations} | "
              f"P1: {cumulative_wins[1]} | P2: {cumulative_wins[-1]} | "
              f"Draw: {cumulative_wins[0]} | "
              f"Avg Scores: P1={np.mean(iter_p1_scores):.1f} P2={np.mean(iter_p2_scores):.1f}")
        
        # Checkpoint
        if iteration % checkpoint_every == 0:
            torch.save({
                'q_net': q_net.state_dict(),
                'model_net': model_net.state_dict(),
                'history': history
            }, f"damath_dynaq_ep{global_step:04d}.pth")
    
    # Final save
    torch.save({
        'q_net': q_net.state_dict(),
        'model_net': model_net.state_dict(),
        'history': history
    }, "DynaQ_Default.pth")
    
    writer.close()
    print(f"🎯 Training complete!")
    return q_net, model_net, agent, history

In [59]:
def play_game_visual(env_factory, agent):
    """Play one game with visualization"""
    env = env_factory()
    move_count = 0
    
    while not env.game_over() and move_count < 300:
        env.print_board()
        print(f"\nPlayer {env.to_move}'s turn. Scores: {env.scores}")
        
        move, action_idx = agent.select_action(env, training=False)
        if move is None:
            break
        
        piece = env.pieces.get(move.path[0])
        if piece:
            piece_type = "Dama" if piece.dama else "Regular"
            print(f"Chosen: {piece_type} {piece.value:+d} at "
                  f"({move.path[0][1]}, {move.path[0][0]}) -> "
                  f"({move.path[-1][1]}, {move.path[-1][0]})")
        
        env.apply_move(move)
        move_count += 1
    
    env.print_board()
    final_scores, winner = env.final_scores_and_winner()
    print(f"\nGame Over! Scores: {final_scores} | Winner: {winner}")
    return winner, final_scores

In [ ]:
# Define env_factory function
def env_factory():
    return DamathEnv(rows=8, cols=8, operator_pattern=operator_pattern_official)

# Train with all the metrics and hyperparameters
q_net, model_net, agent, history = train_dynaq_with_viz(
    env_factory, 
    num_episodes=1000,   # still training for 1000 episodes
    games_per_iter=5,
    epsilon=0.2,         # Exploration Rate
    alpha=0.01,         # Learning Rate
    gamma=0.9,           # Discount Factor
    planning_steps=10
)

# Play a visual game
play_game_visual(env_factory, agent)
plot_training_metrics(history)

🎮 Using device: cuda
📊 Training Configuration:
   Total episodes: 1000
   Games per iteration: 5
   Number of iterations: 200


C:\Users\Coli\AppData\Local\Temp\ipykernel_18500\3980059278.py:245: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(current_q, target_q)


Episode 1/1000 | Winner=1 | Moves=41 | Score: P1=38.0 P2=-17.0 | ε=0.200
Episode 2/1000 | Winner=1 | Moves=66 | Score: P1=63.0 P2=-36.0 | ε=0.200
Episode 3/1000 | Winner=-1 | Moves=35 | Score: P1=-45.0 P2=-8.0 | ε=0.200
Episode 4/1000 | Winner=1 | Moves=46 | Score: P1=108.0 P2=-20.0 | ε=0.200
Episode 5/1000 | Winner=-1 | Moves=58 | Score: P1=-103.0 P2=77.0 | ε=0.200
Additional model training on 246 experiences...
    Epoch 1/3 | State Loss: 0.183 | Reward Loss: 0.045
    Epoch 2/3 | State Loss: 0.150 | Reward Loss: 0.002
    Epoch 3/3 | State Loss: 0.145 | Reward Loss: 0.007
✅ Iter 1/200 | P1: 3 | P2: 2 | Draw: 0 | Avg Scores: P1=12.2 P2=-0.8
Episode 6/1000 | Winner=-1 | Moves=44 | Score: P1=-83.0 P2=13.0 | ε=0.199
Episode 7/1000 | Winner=-1 | Moves=40 | Score: P1=-34.0 P2=34.0 | ε=0.199
Episode 8/1000 | Winner=1 | Moves=33 | Score: P1=139.0 P2=30.0 | ε=0.199
Episode 9/1000 | Winner=1 | Moves=44 | Score: P1=42.0 P2=-12.0 | ε=0.199
Episode 10/1000 | Winner=-1 | Moves=39 | Score: P1=-90.

C:\Users\Coli\AppData\Local\Temp\ipykernel_18500\1142578432.py:63: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  reward_loss = F.mse_loss(pred_rewards.squeeze(), rewards)
C:\Users\Coli\AppData\Local\Temp\ipykernel_18500\1142578432.py:78: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  reward_loss = F.mse_loss(pred_rewards.squeeze(), rewards) # ✅ Reward prediction


Episode 241/1000 | Winner=1 | Moves=56 | Score: P1=30.0 P2=-23.0 | ε=0.152
Episode 242/1000 | Winner=1 | Moves=34 | Score: P1=0.0 P2=-402.0 | ε=0.152
Episode 243/1000 | Winner=1 | Moves=49 | Score: P1=12.0 P2=2.0 | ε=0.152
Episode 244/1000 | Winner=1 | Moves=58 | Score: P1=132.0 P2=-132.0 | ε=0.152
Episode 245/1000 | Winner=-1 | Moves=53 | Score: P1=-37.0 P2=-13.0 | ε=0.152
Additional model training on 250 experiences...
    Epoch 1/3 | State Loss: 0.037 | Reward Loss: 0.067
    Epoch 2/3 | State Loss: 0.027 | Reward Loss: 0.079
    Epoch 3/3 | State Loss: 0.016 | Reward Loss: 0.037
✅ Iter 49/200 | P1: 115 | P2: 128 | Draw: 2 | Avg Scores: P1=27.4 P2=-113.6
Episode 246/1000 | Winner=-1 | Moves=63 | Score: P1=-142.0 P2=16.0 | ε=0.151
Episode 247/1000 | Winner=-1 | Moves=48 | Score: P1=1.0 P2=27.0 | ε=0.151
Episode 248/1000 | Winner=1 | Moves=38 | Score: P1=81.0 P2=35.0 | ε=0.151
Episode 249/1000 | Winner=-1 | Moves=54 | Score: P1=-57.0 P2=63.0 | ε=0.151
Episode 250/1000 | Winner=1 | Mov

In [ ]:
# import torch

# # Method 1: Load the low discount checkpoint
# checkpoint = torch.load('DynaQ_LowDiscount.pth')

# # Method 2: Load the low learning rate checkpoint
# checkpoint = torch.load('DynaQ_LowLearning.pth')

# # See what's inside
# print(checkpoint.keys())
# # Output: dict_keys(['q_net', 'model_net', 'history', 'episodes_completed', ...])


In [ ]:
# def load_trained_agent(checkpoint_path, device=None):
#     """Load a trained DynaQ agent from checkpoint"""
#     if device is None:
#         device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
#     # Load checkpoint
#     checkpoint = torch.load(checkpoint_path, map_location=device)
    
#     # Create networks
#     q_net = QNetwork().to(device)
#     model_net = ModelNetwork().to(device)
    
#     # Load trained weights
#     q_net.load_state_dict(checkpoint['q_net'])
#     model_net.load_state_dict(checkpoint['model_net'])
    
#     # Set to evaluation mode
#     q_net.eval()
#     model_net.eval()
    
#     # Create optimizer (needed for agent initialization)
#     model_optimizer = optim.Adam(model_net.parameters(), lr=0.001)
    
#     # Create agent
#     agent = DynaQAgent(
#         q_net=q_net,
#         model_net=model_net,
#         model_optimizer=model_optimizer,
#         epsilon=0.0,  # No exploration for evaluation
#         alpha=0.001,
#         gamma=0.95,
#         planning_steps=10
#     )
    
#     print(f"✅ Loaded agent from {checkpoint_path}")
#     if 'episodes_completed' in checkpoint:
#         print(f"   Trained on {checkpoint['episodes_completed']} episodes")
    
#     return agent, checkpoint

# # Usage
# agent_low_discount, checkpoint1 = load_trained_agent('DynaQ_LowDiscount.pth')
# agent_low_learning, checkpoint2 = load_trained_agent('DynaQ_LowLearning.pth')

In [ ]:
# # Play a visual game with the loaded agent
# def env_factory():
#     return DamathEnv(rows=8, cols=8, operator_pattern=operator_pattern_official)

# # Test the low discount agent
# print("\n🎮 Testing Low Discount Agent:")
# winner1, scores1 = play_game_visual(env_factory, agent_low_discount)

# # Test the low learning rate agent
# print("\n🎮 Testing Low Learning Rate Agent:")
# winner2, scores2 = play_game_visual(env_factory, agent_low_learning)

In [ ]:
# def agent_vs_agent(env_factory, agent1, agent2, num_games=10):
#     """Have two agents play against each other"""
#     wins = {1: 0, -1: 0, 0: 0}
    
#     for game_num in range(num_games):
#         env = env_factory()
#         move_count = 0
        
#         while not env.game_over() and move_count < 300:
#             # Player 1 uses agent1, Player -1 uses agent2
#             if env.to_move == 1:
#                 move, _ = agent1.select_action(env, training=False)
#             else:
#                 move, _ = agent2.select_action(env, training=False)
            
#             if move is None:
#                 break
            
#             env.apply_move(move)
#             move_count += 1
        
#         _, winner = env.final_scores_and_winner()
#         wins[winner] += 1
#         print(f"Game {game_num+1}: Winner = {winner}")
    
#     print(f"\n📊 Results after {num_games} games:")
#     print(f"   Agent 1 (P1) wins: {wins[1]}")
#     print(f"   Agent 2 (P2) wins: {wins[-1]}")
#     print(f"   Draws: {wins[0]}")
    
#     return wins

# # Compare the two trained agents
# print("\n⚔️ Low Discount vs Low Learning Rate:")
# results = agent_vs_agent(env_factory, agent_low_discount, agent_low_learning, num_games=10)